In [8]:
import requests
import json

url = "https://www.propertyfinder.eg/search/_next/data/w3D6kva46RQ1Q7JDC0Hnc/en/search.json?c=1&fu=0&ob=mr&page=1"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "X-Nextjs-Data": "1",
    "Referer": "https://www.propertyfinder.eg/"
}

try:
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    data = response.json()

    # Grab the listings list array
    listings = data["pageProps"]["searchResult"]["listings"]

    print(f"🎉 Success! Parsing {len(listings)} matched properties using the verified structure:\n")

    for idx, item in enumerate(listings, start=1):
        # Dig into the secondary 'property' nested dictionary block
        prop = item.get("property", {})
        if not prop:
            continue  # Skips item anomalies if it's a promotional banner rather than a property card

        # Basic identification
        property_id = prop.get("id", "N/A")
        title = prop.get("title", "No Title Provided")
        prop_type = prop.get("property_type", "Property")

        # Financial pricing tiers
        price_obj = prop.get("price", {})
        price = price_obj.get("value", 0)
        currency = price_obj.get("currency", "EGP")

        down_payment = prop.get("down_payment_price")
        payment_methods = ", ".join(prop.get("payment_method", []))

        # Precise locations strings
        location_obj = prop.get("location", {})
        location_name = location_obj.get("name", "Unknown Location")
        full_address = location_obj.get("full_name", "Unknown Address")

        # Space dimension metrics
        beds = prop.get("bedrooms", "N/A")
        baths = prop.get("bathrooms", "N/A")

        size_obj = prop.get("size", {})
        size_val = size_obj.get("value", "N/A")
        size_unit = size_obj.get("unit", "sqm")

        # Agencies credentials
        broker_name = prop.get("broker", {}).get("name", "Individual Broker")

        print(f"[{idx}] ID: {property_id} | 🏠 {prop_type}: {title}")
        print(f"    💰 Price: {price:,} {currency}" if isinstance(price, (int, float)) else f"    💰 Total Price: {price} {currency}")
        if down_payment:
            print(f"    💳 Down Payment: {down_payment:,} {currency} ({payment_methods})")
        print(f"    📍 Location: {location_name} ({full_address})")
        print(f"    📐 Specs: {beds} Bed | {baths} Bath | {size_val} {size_unit}")
        print(f"    👤 Agency: {broker_name}")
        print("-" * 75)

except Exception as e:
    print(f"Extraction processing anomaly: {e}")

🎉 Success! Parsing 30 matched properties using the verified structure:

[1] ID: 55002196 | 🏠 Apartment: Below market price ! 1.500.000 Resale installmens
    💰 Price: 5,100,000 EGP
    💳 Down Payment: 3,500,000 EGP (cash, installments)
    📍 Location: Veranda (Veranda, Sahl Hasheesh, Hurghada, Red Sea)
    📐 Specs: studio Bed | 1 Bath | 54 sqm
    👤 Agency: Relax Real Estate
---------------------------------------------------------------------------
[2] ID: 88971734 | 🏠 Apartment: Brand new furnished apartments 1 Min. to the beach
    💰 Price: 9,900,000 EGP
    📍 Location: Bay Village (Bay Village, Sahl Hasheesh, Hurghada, Red Sea)
    📐 Specs: 2 Bed | 1 Bath | 106 sqm
    👤 Agency: Relax Real Estate
---------------------------------------------------------------------------
[3] ID: 88971678 | 🏠 Townhouse: deliver S Villa for sale in Sarai, near Madinaty
    💰 Price: 12,400,000 EGP
    📍 Location: S1 (S1, Sarai, Mostakbal City Compounds, Mostakbal City - Future City, Cairo)
    📐 Specs

In [10]:
import requests
import json
import pandas as pd
import time
import math
import os

OUTPUT_FILE = "property_finder_all_egypt.csv"
BASE_URL = "https://www.propertyfinder.eg/search/_next/data/w3D6kva46RQ1Q7JDC0Hnc/en/search.json"

QUERY_PARAMS = {
    "c": "1",    # 1 = Buy, 2 = Rent
    "fu": "0",
    "ob": "mr",  # Most Recent
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "X-Nextjs-Data": "1",
    "Referer": "https://www.propertyfinder.eg/"
}

def fetch_page_data(page_num):
    params = QUERY_PARAMS.copy()
    params["page"] = str(page_num)
    try:
        response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
        if response.status_code == 404:
            return None
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"\n⚠️ Retrying page {page_num} due to error: {e}")
        time.sleep(4)
        try:
            response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
            return response.json()
        except:
            return None

def extract_total_count(data):
    """Safely extracts overall result count across multiple possible API positions."""
    try:
        # Strategy A: Directly inside searchResult block
        return data["pageProps"]["searchResult"]["res_count"]
    except KeyError:
        try:
            # Strategy B: Stored inside layout/meta variables
            return data["pageProps"]["meta"]["total_count"]
        except KeyError:
            return None

def parse_listings(listings_data):
    extracted = []
    for item in listings_data:
        prop = item.get("property", {})
        if not prop:
            continue

        extracted.append({
            "ID": prop.get("id"),
            "Type": prop.get("property_type"),
            "Title": prop.get("title"),
            "Price (EGP)": prop.get("price", {}).get("value"),
            "Down Payment": prop.get("down_payment_price"),
            "Location Cluster": prop.get("location", {}).get("name"),
            "Full Address": prop.get("location", {}).get("full_name"),
            "Bedrooms": prop.get("bedrooms"),
            "Bathrooms": prop.get("bathrooms"),
            "Area (sqm)": prop.get("size", {}).get("value"),
            "Agency Name": prop.get("broker", {}).get("name"),
            "URL": prop.get("share_url")
        })
    return extracted

def scrape_real_estate():
    print("🚀 Initializing dynamic full-site scraping sequence...")

    # Run a direct probe on Page 1 to decode sizing layout parameters
    first_page_raw = fetch_page_data(1)
    if not first_page_raw:
        print("❌ Fatal: Could not connect to the target endpoint structure.")
        return

    total_properties = extract_total_count(first_page_raw)

    # Fallback to handle edge cases if the structural count is hidden
    if not total_properties:
        print("⚠️ Summary metadata field skipped by host. Defaulting to system pagination depth limit.")
        total_properties = 179424  # Use site baseline

    items_per_page = 30
    calculated_max_pages = math.ceil(total_properties / items_per_page)

    # Enforce API protection barrier. Real estate engines cut listings off around page 100 per segment
    max_target_pages = min(calculated_max_pages, 100)

    print(f"📊 Live System Assessment:")
    print(f"   • Total Active Properties on Site: {total_properties:,}")
    print(f"   • Page Target Allocation Limit: {max_target_pages} pages (~3,000 properties this loop)")
    print("   💡 Note: To capture all 179k+, we'll run targeted geographic filter passes later.")
    print("-" * 75)

    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)

    total_saved = 0

    for current_page in range(1, max_target_pages + 1):
        print(f"🔄 Processing: Page {current_page}/{max_target_pages}...", end="", flush=True)

        page_json = fetch_page_data(current_page)
        if not page_json:
            print(" 🛑 [Blocked or Limit Met]")
            break

        try:
            # Safely grab whatever listing node exists
            search_obj = page_json.get("pageProps", {}).get("searchResult", {})
            raw_listings = search_obj.get("listings", [])

            if not raw_listings:
                print(" ✅ [Reached End of Feed]")
                break

            parsed_properties = parse_listings(raw_listings)

            if parsed_properties:
                df = pd.DataFrame(parsed_properties)
                df.to_csv(OUTPUT_FILE, mode='a', index=False, header=not os.path.exists(OUTPUT_FILE), encoding="utf-8-sig")
                total_saved += len(parsed_properties)
                print(f" Cleaned & Saved +{len(parsed_properties)} items. (Total: {total_saved:,})")
            else:
                print(" ⚠️ Empty array container.")

        except Exception as e:
            print(f" ❌ Parsing error: {e}")
            break

        time.sleep(2.0) # Throttling step to keep your signature natural

    print(f"\n🎉 Extraction Cycle Complete! Total properties archived: {total_saved:,}")
    print(f"💾 File written to target file block: '{OUTPUT_FILE}'")

if __name__ == "__main__":
    scrape_real_estate()

🚀 Initializing dynamic full-site scraping sequence...
⚠️ Summary metadata field skipped by host. Defaulting to system pagination depth limit.
📊 Live System Assessment:
   • Total Active Properties on Site: 179,424
   • Page Target Allocation Limit: 100 pages (~3,000 properties this loop)
   💡 Note: To capture all 179k+, we'll run targeted geographic filter passes later.
---------------------------------------------------------------------------
🔄 Processing: Page 1/100... Cleaned & Saved +25 items. (Total: 25)
🔄 Processing: Page 2/100... Cleaned & Saved +25 items. (Total: 50)
🔄 Processing: Page 3/100... Cleaned & Saved +25 items. (Total: 75)
🔄 Processing: Page 4/100... Cleaned & Saved +25 items. (Total: 100)
🔄 Processing: Page 5/100...
⚠️ Retrying page 5 due to error: 405 Client Error: Not Allowed for url: https://www.propertyfinder.eg/search/_next/data/w3D6kva46RQ1Q7JDC0Hnc/en/search.json?c=1&fu=0&ob=mr&page=5
 🛑 [Blocked or Limit Met]

🎉 Extraction Cycle Complete! Total properties ar

In [15]:
import time
import csv
import json
import logging
import httpx
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

OUTPUT_FILE = "propertyfinder_egypt_master.csv"

# Enhanced headers mimicking standard browser traffic configuration
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive"
}

def parse_property(item_node):
    """Safely extracts inner data arrays from the verified property dictionary structure."""
    # Dig down into the secondary nested 'property' node revealed by the data layout probe
    prop = item_node.get("property", {})
    if not prop:
        return None

    try:
        price_obj = prop.get("price", {})
        size_obj = prop.get("size", {})

        return {
            "id": prop.get("id"),
            "type": prop.get("property_type"),
            "title": prop.get("title"),
            "price": price_obj.get("value") if isinstance(price_obj, dict) else price_obj,
            "currency": price_obj.get("currency", "EGP") if isinstance(price_obj, dict) else "EGP",
            "down_payment": prop.get("down_payment_price", "N/A"),
            "location_name": prop.get("location", {}).get("name") if isinstance(prop.get("location"), dict) else "N/A",
            "full_address": prop.get("location", {}).get("full_name") if isinstance(prop.get("location"), dict) else "N/A",
            "beds": prop.get("bedrooms"),
            "baths": prop.get("bathrooms"),
            "area_sqm": size_obj.get("value") if isinstance(size_obj, dict) else prop.get("size"),
            "broker": prop.get("broker", {}).get("name", "N/A") if isinstance(prop.get("broker"), dict) else "N/A",
            "url": prop.get("share_url", "")
        }
    except Exception as e:
        logging.warning(f"Skipping anomaly element node parsing error: {e}")
        return None

def main():
    # Force HTTP/2 tracking layers to blend into standard device metrics footprints
    # Ensure you ran `pip install httpx[http2]` prior to execution to support this context library
    with httpx.Client(http2=True, headers=HEADERS, timeout=30.0) as client:

        with open(OUTPUT_FILE, mode="w", newline="", encoding="utf-8") as f:
            fields = ["id", "type", "title", "price", "currency", "down_payment", "location_name", "full_address", "beds", "baths", "area_sqm", "broker", "url"]
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()

            total_records = 0
            page = 1

            # Absolute processing barrier limit set to page 100 to respect platform constraints
            max_pages = 100

            logging.info("🚀 Starting Master Extraction Pipeline across Egypt data stream...")

            while page <= max_pages:
                # Target the public standard search result parameters natively
                target_url = f"https://www.propertyfinder.eg/en/buy/properties-for-sale.html?ob=mr&page={page}"
                logging.info(f"Processing: Page {page}/{max_pages}...")

                try:
                    res = client.get(target_url)

                    if res.status_code != 200:
                        logging.error(f"Execution blocked with server response code: {res.status_code}")
                        break

                    soup = BeautifulSoup(res.text, "html.parser")
                    script_block = soup.find("script", id="__NEXT_DATA__")

                    if not script_block:
                        logging.error("Next.js hydration element missing from page code structure.")
                        break

                    data = json.loads(script_block.string)
                    search_result = data.get("props", {}).get("pageProps", {}).get("searchResult", {})

                    # Core fix: Point extraction tracking explicitly to the verified 'listings' array key path
                    listings = search_result.get("listings", [])

                    if not listings:
                        logging.info("✅ Reached data boundary threshold. All items captured.")
                        break

                    saved_on_page = 0
                    for item in listings:
                        record = parse_property(item)
                        if record:
                            writer.writerow(record)
                            saved_on_page += 1
                            total_records += 1

                    logging.info(f"   ↳ Stored +{saved_on_page} items this loop iteration. (Total Archive: {total_records:,})")

                    # Update iteration dynamic limits if the server updates totals during execution
                    server_page_max = search_result.get("pageCount")
                    if server_page_max and server_page_max < max_pages:
                        max_pages = server_page_max

                    page += 1
                    time.sleep(2.0)  # Safe delay sequence parameter throttle

                except Exception as e:
                    logging.error(f"Fatal skip occurred during runtime loop processing: {e}")
                    break

        logging.info(f"🎯 Execution Complete. Master data committed to '{OUTPUT_FILE}'. Captured {total_records:,} units.")

if __name__ == "__main__":
    main()

2026-07-05 23:35:54,205 - INFO - 🚀 Starting Master Extraction Pipeline across Egypt data stream...
2026-07-05 23:35:54,206 - INFO - Processing: Page 1/100...
2026-07-05 23:35:55,031 - INFO - HTTP Request: GET https://www.propertyfinder.eg/en/buy/properties-for-sale.html?ob=mr&page=1 "HTTP/2 200 OK"
2026-07-05 23:35:55,438 - INFO -    ↳ Stored +25 items this loop iteration. (Total Archive: 25)
2026-07-05 23:35:57,439 - INFO - Processing: Page 2/100...
2026-07-05 23:35:57,902 - INFO - HTTP Request: GET https://www.propertyfinder.eg/en/buy/properties-for-sale.html?ob=mr&page=2 "HTTP/2 200 OK"
2026-07-05 23:35:58,119 - INFO -    ↳ Stored +25 items this loop iteration. (Total Archive: 50)
2026-07-05 23:36:00,120 - INFO - Processing: Page 3/100...
2026-07-05 23:36:00,662 - INFO - HTTP Request: GET https://www.propertyfinder.eg/en/buy/properties-for-sale.html?ob=mr&page=3 "HTTP/2 200 OK"
2026-07-05 23:36:00,910 - INFO -    ↳ Stored +25 items this loop iteration. (Total Archive: 75)
2026-07-05